# Introduction

Aujourd'hui la plupart des sites web utilisent du JavaScript pour rendre le site plus dynamique et agréable, mais aussi pour afficher la donnée. 

Pour ces sites, la première méthode abordée n'est pas efficace, car de nombreuses requêtes sont utilisées pour se connecter, afficher la donnée, naviguer, il faudrait prendre en compte toutes ces requêtes et les assembler pour simuler le comportement du site. 

**C'est le comportement de votre navigateur**, il compile toutes les interactions, les requêtes et génère du HTML permettant d'afficher l'interface graphique d'un site web.

Pour palier ce problème, on peut laisser le navigateur faire son travail et contrôler sa surcouche logicielle. Pour cela, on peut prendre le contrôle d'un navigateur comme Chrome, Edge, Firefox ou Safari depuis un script Python. 

Pour cela, nous allons utiliser un package Python `Selenium` https://selenium-python.readthedocs.io/ qui permet d'instancier un navigateur et de le contrôler. Sélénium est beaucoup utilisé pour générer des tests automatiques de sites web. 

On peut trouver les drivers des différents browser : 
- Chrome:	https://googlechromelabs.github.io/chrome-for-testing/#stable (bien télécharger le "chromedriver" et non pas chrome)
- Firefox:	https://github.com/mozilla/geckodriver/releases
- Safari:	https://webkit.org/blog/6900/webdriver-support-in-safari-10/

**Depuis les dernières versions de Selenium, il n'est plus néccessaires de télécharger les driver des browsers car Selenium les manage**

In [1]:
from selenium import webdriver


In [3]:
chrome = webdriver.Chrome()

Vous devez voir apparaître une nouvelle instance de votre navigateur.
Chrome vous demande depuis peu de selectionner votre navigateur par défaut. Etant donné que notre but est de tout controller depuis notre code, on peut bypass cette demande en ajoutant des options au webdriver de Chrome

In [4]:
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")

chrome = webdriver.Chrome(options=chrome_options)

Maintenant, vous avez le controlle du navigateur depuis Python. Si vous voulez accéder à une page :

In [5]:
chrome.get("https://books.toscrape.com/")

Allez maintenant voir votre fenêtre. Vous êtes sur un faux site de vente de livre. Ce site est spécialement concu pour être scrapé et comprendre les bases, mais en réalité scraper un site est un peu plus complexe, comme vous allez vous en rendre compte dans la section 2 sur Scrapy.

Vous pouvez maintenant vous déplacer dans la page. Pour cela, il va falloir inspecter le code source de la page.

Il existe de nombreuse méthode pour récupérer les éléments par exemple `find_element` qui permet de récupérer un élément grâce au texte affiché. Ici, si on veut récupérer cliquer sur la section `Travel` pour récupérer les livres de voyages

In [6]:
from selenium.webdriver.common.by import By
link = chrome.find_element(By.LINK_TEXT, "Travel")
type(link)

selenium.webdriver.remote.webelement.WebElement

Si on veut entrer dans le lien, il suffit d'appeler la méthode `click`.

In [7]:
link.click()

Vous êtes arrivé sur la page des livres de voyage. Maintenant nous allons récupérer toutes les livres. On remarque que tous les livres ont une class HTML nommée `product_pod`.

In [8]:
all_widgets = chrome.find_elements(By.CLASS_NAME, "product_pod")
type(all_widgets), len(all_widgets)

(list, 11)

On obtient une liste d'éléments décrits par cette class. Il y en a 11, un pour chaque livre. Pour récupérer le titre du premier livre: 

In [9]:
first_book = all_widgets[0]
first_book_title = first_book.find_element(By.CSS_SELECTOR, "h3")
first_book_title.text

"It's Only the Himalayas"

Si vous êtes fan de JavaScript vous pouvez même injecter du code JS dans le navigateur depuis Python pour exécuter des opérations complexes. 

In [10]:
js_script = """
const class_name = 'DSIA'
alert(`Hi from ${class_name}`)
"""

In [11]:
chrome.execute_script(js_script)

# Exercice

## Exercice 1

Extraire le prix des livres :

In [12]:
def extract_price(book_web_element):
    price = book_web_element.find_element(
        By.CLASS_NAME, "price_color"
    )
    return price.text


extract_price(all_widgets[0])

UnexpectedAlertPresentException: Alert Text: Hi from DSIA
Message: unexpected alert open: {Alert text : Hi from DSIA}
  (Session info: chrome=157.0.8103.79)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff6523ad655]
	(No symbol) [0x7ff6522a4790]
	(No symbol) [0x7ff652112fb3]
	(No symbol) [0x7ff652147a25]
	(No symbol) [0x7ff65212c109]
	(No symbol) [0x7ff65212d433]
	(No symbol) [0x7ff652847041]
	(No symbol) [0x7ff652865c76]
	(No symbol) [0x7ff65285c340]
	(No symbol) [0x7ff6524dc9be]
	(No symbol) [0x7ff6522b2fa5]
	(No symbol) [0x7ff6522ae2d4]
	(No symbol) [0x7ff6522ae464]
	(No symbol) [0x7ff65229731c]
	(No symbol) [0x7ffa7eb9259d]
	(No symbol) [0x7ffa8062af78]


Extraire l'image des livres: 

In [ ]:
def extract_image(book_web_element):
    image = book_web_element.find_element(
        By.TAG_NAME, "img"
    )
    return image.get_attribute("src")


extract_image(all_widgets[0])

NotImplementedError: Need to be implemented

Extraire le nombre d'étoiles d'un livre: 

In [13]:
def extract_rating(book_web_element):
    rating = book_web_element.find_element(
        By.CLASS_NAME, "star-rating"
    )

    stars = rating.get_attribute("class").split()[1]

    ratings = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    return ratings[stars]


extract_rating(all_widgets[0])

2

In [14]:
import pandas as pd

books = []

for book in all_widgets:

    books.append({
        "titre": book.find_element(
            By.CSS_SELECTOR, "h3 a"
        ).get_attribute("title"),
        "prix": extract_price(book),
        "image": extract_image(book),
        "note": extract_rating(book)
    })

df = pd.DataFrame(books)

df

NameError: name 'extract_image' is not defined

In [46]:
chrome.close()